# Clean ATP UK Data

Load and clean UK Data

## Set Up Notebook

In [293]:
# Import Packages
from pathlib import Path
import pandas as pd

# Import Local Package
from tennis_data_pipeline.handler.uk.validatior.tournaments import (
    find_uk_inconsistent_tournaments, 
    find_uk_reused_tournament_ids, 
)
from tennis_data_pipeline.cleaner.uk import atp_cols as cols
from tennis_data_pipeline.cleaner.uk.atp import(
    build_uk_atp_quality_report,
    clean_uk_atp_data,
    summarize_uk_atp_quality,
)


# Constants
YEAR = 2010
# YEAR = int(input("Enter the year: "))
PROJECT_DIR = Path("../..").resolve()
print("Base Directory:", PROJECT_DIR)


Base Directory: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline


### Constants

In [294]:
def load_dirty_uk_atp_data(year: int) -> pd.DataFrame:
    df_uk = pd.read_csv(PROJECT_DIR / f"data/raw/tennis-data-uk/2024-10/atp/atp_singles_results_{year}.csv")

    # Date format drifts across seasons (e.g. "1/1/23" vs. "2000-01-03").
    df_uk["Date"] = pd.to_datetime(df_uk["Date"], format="mixed", dayfirst=False)
    df_uk["Year"] = year

    # Nullable ints: ranks/points/set-scores are whole numbers but can be missing (e.g. retired matches).
    int_cols = [
        "ATP", "Year", "Best of",
        "WRank", "LRank", "WPts", "LPts",
        "W1", "L1", "W2", "L2", "W3", "L3", "W4", "L4", "W5", "L5",
        "Wsets", "Lsets",
    ]
    for col in int_cols:
        if col in df_uk.columns:
            df_uk[col] = pd.to_numeric(df_uk[col], errors="coerce").astype("Int64")

    # Everything left over is bookmaker odds; which bookmakers are present varies by year.
    known_cols = {
        "ATP", "Year", "Location", "Tournament", "Date", "Series", "Court",
        "Surface", "Round", "Best of", "Winner", "Loser", "Comment", *int_cols,
    }
    odds_cols = [col for col in df_uk.columns if col not in known_cols]
    df_uk[odds_cols] = df_uk[odds_cols].apply(pd.to_numeric, errors="coerce")

    for col in ["Series", "Court", "Surface", "Round", "Comment"]:
        if col in df_uk.columns:
            df_uk[col] = df_uk[col].astype("category")

    return df_uk


## Load Data

In [295]:
df_uk_raw = load_dirty_uk_atp_data(YEAR)
df_uk = df_uk_raw.copy()
print(list(df_uk.columns))

print(df_uk.head())


['ATP', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'EXW', 'EXL', 'LBW', 'LBL', 'PSW', 'PSL', 'SJW', 'SJL', 'MaxW', 'MaxL', 'AvgW', 'AvgL', 'Year']
   ATP  Location              Tournament       Date  Series    Court Surface  \
0    1  Brisbane  Brisbane International 2010-01-04  ATP250  Outdoor    Hard   
1    1  Brisbane  Brisbane International 2010-01-04  ATP250  Outdoor    Hard   
2    1  Brisbane  Brisbane International 2010-01-04  ATP250  Outdoor    Hard   
3    1  Brisbane  Brisbane International 2010-01-04  ATP250  Outdoor    Hard   
4    1  Brisbane  Brisbane International 2010-01-04  ATP250  Outdoor    Hard   

       Round  Best of      Winner  ...    LBL    PSW    PSL    SJW    SJL  \
0  1st Round        3  Gasquet R.  ...  2.375  1.526  2.740  1.500  2.500   
1  1st Round     

## Check Tournaments

In [296]:
ATP_BEST_OF_5_TOURNAMENTS = {
    "Australian Open",
    "French Open",
    "Roland Garros",
    "Wimbledon",
    "US Open",
}

# All ATP Grand Slams are best-of-5; the raw source mislabels some individual matches.
grand_slam_mask = df_uk["Tournament"].isin(ATP_BEST_OF_5_TOURNAMENTS)
df_uk.loc[grand_slam_mask, "Best of"] = 5

# ATP Finals (Masters Cup) is best-of-3 for every round; the raw source omits
# "Best of" entirely for these rows rather than mislabeling it.
masters_cup_mask = df_uk["Series"] == "Masters Cup"
df_uk.loc[masters_cup_mask & df_uk["Best of"].isna(), "Best of"] = 3


In [297]:
years_with_known_tourney_inconsistencies = [2023]
if YEAR not in years_with_known_tourney_inconsistencies:
    inconsistent_tourneys = find_uk_inconsistent_tournaments(df_uk, key_columns=["ATP", "Year", "Location"], info_cols=["Tournament", "Series", "Court", "Surface", "Best of"])
    print("Inconsistent Tournaments:")
    print(inconsistent_tourneys[0])

    print("Rows Effected")
    display(inconsistent_tourneys[1])

    if not inconsistent_tourneys[0].empty and not inconsistent_tourneys[1].empty:
        raise ValueError("There are inconsistent tournaments with no affected rows.")

All tournament attributes are consistent.
Inconsistent Tournaments:
Empty DataFrame
Columns: [Tournament, Series, Court, Surface, Best of]
Index: []
Rows Effected


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,LBL,PSW,PSL,SJW,SJL,MaxW,MaxL,AvgW,AvgL,Year


In [298]:
reused_tourney_id = find_uk_reused_tournament_ids(df=df_uk, id_col= "ATP", disambiguating_cols=["Location", "Tournament"])
print("ATP Reused Tournament IDs")
display(reused_tourney_id[0])

print("Rows Affected")
display(reused_tourney_id[1])

if not reused_tourney_id[0].empty and not reused_tourney_id[1].empty:
    raise ValueError("There are reused tournament IDs with no affected rows.")

Every ATP id maps to exactly one tournament.
ATP Reused Tournament IDs


,Location,Tournament
ATP,,


Rows Affected


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,LBL,PSW,PSL,SJW,SJL,MaxW,MaxL,AvgW,AvgL,Year


In [299]:
# df_uk.loc[df_uk["Location"] == "Montpellier"]

### Bad Odds Checker

In [300]:
RAW_ODDS_COLS = ["B365W", "B365L", "PSW", "PSL", "MaxW", "MaxL", "AvgW", "AvgL"]

# Decimal odds are always >= 1.0; the raw source has occasional entry errors (e.g. a
# misplaced decimal point) that produce an impossible odd below 1. Capture the
# original rows for review before nulling them out (rather than guessing the
# intended value).
bad_odds_mask = (df_uk[RAW_ODDS_COLS] < 1).any(axis=1)
bad_odds_rows = df_uk.loc[
    bad_odds_mask,
    ["ATP", "Location", "Tournament", "Round", "Winner", "Loser", "Comment", *RAW_ODDS_COLS],
]

for col in RAW_ODDS_COLS:
    if col in df_uk.columns:
        bad_odds = df_uk[col] < 1
        if bad_odds.any():
            print(f"{col}: nulling {bad_odds.sum()} odds < 1")
            df_uk.loc[bad_odds, col] = pd.NA

print(bad_odds_rows)

Empty DataFrame
Columns: [ATP, Location, Tournament, Round, Winner, Loser, Comment, B365W, B365L, PSW, PSL, MaxW, MaxL, AvgW, AvgL]
Index: []


In [301]:
print(f"{len(bad_odds_rows)} row(s) had an odd below 1.0 (nulled above):")
display(bad_odds_rows)


0 row(s) had an odd below 1.0 (nulled above):


,ATP,Location,Tournament,Round,Winner,Loser,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL


### Bad Set-Score Checker

In [302]:
if YEAR == 2019:
    # 2019 Metz final (Tsonga d. Bedene): raw source has a corrupted set-2 score and
    # is missing set 3 entirely, so Wsets/Lsets (0-1) contradict the "Completed" status.
    # Verified actual result was 6-7, 7-6, 6-3 to Tsonga (en.wikipedia.org/wiki/2019_Moselle_Open).
    metz_final_mask = (
        (df_uk["ATP"] == 53)
        & (df_uk["Winner"] == "Tsonga J.W.")
        & (df_uk["Loser"] == "Bedene A.")
        & (df_uk["Date"] == "2019-09-22")
    )
    print(f"Rows matched: {metz_final_mask.sum()}")

    df_uk.loc[metz_final_mask, ["W1", "L1", "W2", "L2", "W3", "L3"]] = [6, 7, 7, 6, 6, 3]
    df_uk.loc[metz_final_mask, ["Wsets", "Lsets"]] = [2, 1]

if YEAR == 2015:
    # 2015 Nottingham (AEGON Open) semifinal (Istomin d. Baghdatis): Wikipedia lists
    # Baghdatis under the tournament's retirements, but the raw source labels this
    # "Completed" despite only a single incomplete set (1-2) being recorded.
    nottingham_sf_mask = (
        (df_uk["ATP"] == 38)
        & (df_uk["Winner"] == "Istomin D.")
        & (df_uk["Loser"] == "Baghdatis M.")
        & (df_uk["Date"] == "2015-06-26")
    )
    print(f"Rows matched: {nottingham_sf_mask.sum()}")

    df_uk.loc[nottingham_sf_mask, "Comment"] = "Retired"

if YEAR == 2013:
    # 2013 Bogota (Claro Open Colombia) final (Karlovic d. Falla): raw source has
    # Wsets/Lsets both recorded as 0 despite two straight sets being recorded.
    # Verified actual result was 6-3, 7-6 to Karlovic (en.wikipedia.org/wiki/2013_Claro_Open_Colombia).
    bogota_final_mask = (
        (df_uk["ATP"] == 41)
        & (df_uk["Winner"] == "Karlovic I.")
        & (df_uk["Loser"] == "Falla A.")
        & (df_uk["Date"] == "2013-07-21")
    )
    print(f"Rows matched: {bogota_final_mask.sum()}")

    df_uk.loc[bogota_final_mask, ["Wsets", "Lsets"]] = [2, 0]


## Clean Data

In [303]:
df_uk = clean_uk_atp_data(df_uk)

Sanity-check renamed categories

In [304]:
for col in [
    "series",
    "surface",
    "round",
    "match_status",
]:
    print(f"\n{col}")
    print(
        df_uk[col]
        .value_counts(dropna=False)
        .sort_index()
    )

print("\nis_outdoor")
print(df_uk["is_outdoor"].value_counts(dropna=False))


series
series
atp_250         1192
atp_500          397
grand_slam       508
masters_1000     567
tour_finals       15
Name: count, dtype: int64

indoor_outdoor
indoor_outdoor
indoor      495
outdoor    2184
Name: count, dtype: int64

surface
surface
clay      823
grass     302
hard     1554
Name: count, dtype: int64

round
round
F        65
QF      256
R128    320
R16     512
R32     936
R64     448
RR       12
SF      130
Name: count, dtype: int64

match_status
match_status
completed    2569
retired        94
walkover       16
Name: count, dtype: int64


## Explore Data Quality

In [305]:
summarize_uk_atp_quality(df_uk)

Rows: 2679
Duplicate match keys: 0

Match status:
match_status
completed    2569
retired        94
walkover       16
Name: count, dtype: int64

Missingness:
winner_set_5_games      96.229937
loser_set_5_games       96.229937
loser_set_4_games       90.892124
winner_set_4_games      90.892124
winner_set_3_games      55.207167
loser_set_3_games       55.207167
odds_max_winner         35.423666
odds_max_loser          35.423666
odds_avg_winner         35.423666
odds_avg_loser          35.423666
loser_set_2_games        1.717059
winner_set_2_games       1.717059
odds_pinnacle_winner     1.007839
odds_pinnacle_loser      1.007839
odds_b365_winner         0.895857
odds_b365_loser          0.821202
loser_sets               0.597238
loser_set_1_games        0.597238
winner_set_1_games       0.597238
winner_sets              0.597238
loser_rank_points        0.186637
loser_rank               0.186637
winner_rank              0.037327
winner_rank_points       0.037327
dtype: float64

Completed m

In [306]:
score_cols = [
    "winner_set_1_games",
    "loser_set_1_games",
    "winner_set_2_games",
    "loser_set_2_games",
    "winner_sets",
    "loser_sets",
]

for col in score_cols:
    print(f"\n{col}")
    print(
        df_uk.loc[df_uk[col].isna(), "match_status"]
        .value_counts(dropna=False)
    )


winner_set_1_games
match_status
walkover     16
completed     0
retired       0
Name: count, dtype: int64

loser_set_1_games
match_status
walkover     16
completed     0
retired       0
Name: count, dtype: int64

winner_set_2_games
match_status
retired      29
walkover     16
completed     1
Name: count, dtype: int64

loser_set_2_games
match_status
retired      29
walkover     16
completed     1
Name: count, dtype: int64

winner_sets
match_status
walkover     16
completed     0
retired       0
Name: count, dtype: int64

loser_sets
match_status
walkover     16
completed     0
retired       0
Name: count, dtype: int64


In [307]:
missing_first_set = df_uk.loc[
    df_uk["winner_set_1_games"].isna()
    | df_uk["loser_set_1_games"].isna(),
    [
        "match_date",
        "tournament_name",
        "round",
        "winner_name",
        "loser_name",
        "match_status",
        "winner_set_1_games",
        "loser_set_1_games",
        "winner_sets",
        "loser_sets",
    ],
]

print(missing_first_set)

     match_date                             tournament_name round  \
251  2010-01-23                             Australian Open   R32   
422  2010-02-12            ABN AMRO World Tennis Tournament    QF   
484  2010-02-19                                 Copa Telmex    QF   
599  2010-02-25                 International Championships   R16   
686  2010-03-14                            BNP Paribas Open   R64   
706  2010-03-15                            BNP Paribas Open   R32   
716  2010-03-16                            BNP Paribas Open   R32   
1040 2010-04-29                 Internazionali BNL d'Italia   R16   
1421 2010-06-10                         AEGON Championships   R16   
1569 2010-06-24                                   Wimbledon   R64   
2007 2010-08-19  Western & Southern Financial Group Masters   R16   
2243 2010-09-25                             Open de Moselle    SF   
2318 2010-10-06                                  China Open   R16   
2354 2010-10-08     Rakuten Japan 

In [308]:
missing_odds = df_uk.loc[
    df_uk[cols.ODDS_COLS].isna().any(axis=1),
    [
        "match_date",
        "tournament_name",
        "round",
        "winner_name",
        "loser_name",
        "match_status",
        *cols.ODDS_COLS,
    ],
]

print(missing_odds)

     match_date         tournament_name round    winner_name  \
0    2010-01-04  Brisbane International   R32     Gasquet R.   
1    2010-01-04  Brisbane International   R32     Odesnik W.   
2    2010-01-04  Brisbane International   R32     Gicquel M.   
3    2010-01-04  Brisbane International   R32       Falla A.   
4    2010-01-04  Brisbane International   R32        Levy H.   
...         ...                     ...   ...            ...   
2395 2010-10-13        Shanghai Masters   R32    Tsonga J.W.   
2396 2010-10-13        Shanghai Masters   R32      Ferrer D.   
2397 2010-10-13        Shanghai Masters   R32       Mayer F.   
2419 2010-10-19             Kremlin Cup   R32  Dolgopolov O.   
2600 2010-11-03       Valencia Open 500   R32      Cuevas P.   

            loser_name match_status  odds_b365_winner  odds_b365_loser  \
0          Nieminen J.    completed              1.44             2.62   
1           Clement A.    completed              2.25             1.57   
2        

## Quality Report

In [309]:
quality_report = build_uk_atp_quality_report(df_uk)
print(quality_report.T)


                                  Metric_2010
year                              2010.000000
rows                              2679.000000
duplicate_match_keys                 0.000000
status_count_completed            2569.000000
status_count_retired                94.000000
status_count_walkover               16.000000
completed_missing_odds             924.000000
missing_pct_winner_rank              0.037327
missing_pct_loser_rank               0.186637
missing_pct_winner_rank_points       0.037327
missing_pct_loser_rank_points        0.186637
missing_pct_winner_sets              0.597238
missing_pct_loser_sets               0.597238
missing_pct_winner_set_1_games       0.597238
missing_pct_loser_set_1_games        0.597238
missing_pct_winner_set_2_games       1.717059
missing_pct_loser_set_2_games        1.717059
missing_pct_winner_set_3_games      55.207167
missing_pct_loser_set_3_games       55.207167
missing_pct_winner_set_4_games      90.892124
missing_pct_loser_set_4_games     

In [310]:
quality_report.T

,Metric_2010
year,2010.000000
rows,2679.000000
duplicate_match_keys,0.000000
status_count_completed,2569.000000
status_count_retired,94.000000
status_count_walkover,16.000000
completed_missing_odds,924.000000
missing_pct_winner_rank,0.037327
missing_pct_loser_rank,0.186637
missing_pct_winner_rank_points,0.037327


In [311]:
# Write to quality report
quality_report_path = (
    PROJECT_DIR
    / "data/clean/tennis-data-uk/atp/analysis/uk_atp_quality_report.csv"
)
quality_report_path.parent.mkdir(parents=True, exist_ok=True)

# Accumulate one row per year across notebook runs;
# re-running a year overwrites its old row.
if quality_report_path.exists():
    existing_report = pd.read_csv(
        quality_report_path,
        index_col=0,
    )

    combined_report = pd.concat(
        [existing_report, quality_report]
    )

    combined_report = combined_report[
        ~combined_report.index.duplicated(keep="last")
    ]
else:
    combined_report = quality_report


# Metrics where a missing value really means "zero"
zero_fill_cols = [
    col
    for col in combined_report.columns
    if (
        col.startswith("status_count_")
        or col.startswith("missing_pct_")
        or col
        in {
            "duplicate_match_keys",
            "completed_missing_odds",
        }
    )
]

combined_report[zero_fill_cols] = (
    combined_report[zero_fill_cols]
    .fillna(0)
)

combined_report = combined_report.sort_values("year")
combined_report = combined_report.round(4)

combined_report.index.name = "index"

combined_report.to_csv(quality_report_path)

print(f"Written to {quality_report_path}")

Written to /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/clean/tennis-data-uk/atp/analysis/uk_atp_quality_report.csv


In [312]:
print(list(df_uk.columns))

print(df_uk.head())

['source', 'tour', 'year', 'uk_tournament_id', 'tournament_name', 'location', 'match_date', 'series', 'indoor_outdoor', 'surface', 'round', 'best_of', 'winner_name', 'loser_name', 'winner_rank', 'loser_rank', 'winner_rank_points', 'loser_rank_points', 'winner_sets', 'loser_sets', 'winner_set_1_games', 'loser_set_1_games', 'winner_set_2_games', 'loser_set_2_games', 'winner_set_3_games', 'loser_set_3_games', 'winner_set_4_games', 'loser_set_4_games', 'winner_set_5_games', 'loser_set_5_games', 'match_status', 'odds_b365_winner', 'odds_b365_loser', 'odds_pinnacle_winner', 'odds_pinnacle_loser', 'odds_max_winner', 'odds_max_loser', 'odds_avg_winner', 'odds_avg_loser', 'source_event_key', 'source_match_key']
           source tour  year  uk_tournament_id         tournament_name  \
0  tennis_data_uk  atp  2010                 1  Brisbane International   
1  tennis_data_uk  atp  2010                 1  Brisbane International   
2  tennis_data_uk  atp  2010                 1  Brisbane Internati

In [313]:
display(df_uk)

,source,tour,year,uk_tournament_id,tournament_name,location,match_date,series,indoor_outdoor,surface,...,odds_b365_winner,odds_b365_loser,odds_pinnacle_winner,odds_pinnacle_loser,odds_max_winner,odds_max_loser,odds_avg_winner,odds_avg_loser,source_event_key,source_match_key
0,tennis_data_uk,atp,2010,1,Brisbane International,Brisbane,2010-01-04,atp_250,outdoor,hard,...,1.44,2.62,1.526,2.740,NaN,NaN,NaN,NaN,2010_1_brisbane_brisbane_international,2010_1_brisbane_brisbane_international_2010-01...
1,tennis_data_uk,atp,2010,1,Brisbane International,Brisbane,2010-01-04,atp_250,outdoor,hard,...,2.25,1.57,2.140,1.813,NaN,NaN,NaN,NaN,2010_1_brisbane_brisbane_international,2010_1_brisbane_brisbane_international_2010-01...
2,tennis_data_uk,atp,2010,1,Brisbane International,Brisbane,2010-01-04,atp_250,outdoor,hard,...,1.61,2.20,1.676,2.360,NaN,NaN,NaN,NaN,2010_1_brisbane_brisbane_international,2010_1_brisbane_brisbane_international_2010-01...
3,tennis_data_uk,atp,2010,1,Brisbane International,Brisbane,2010-01-04,atp_250,outdoor,hard,...,2.62,1.44,2.580,1.581,NaN,NaN,NaN,NaN,2010_1_brisbane_brisbane_international,2010_1_brisbane_brisbane_international_2010-01...
4,tennis_data_uk,atp,2010,1,Brisbane International,Brisbane,2010-01-04,atp_250,outdoor,hard,...,3.00,1.36,2.170,1.787,NaN,NaN,NaN,NaN,2010_1_brisbane_brisbane_international,2010_1_brisbane_brisbane_international_2010-01...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2674,tennis_data_uk,atp,2010,65,Masters Cup,London,2010-11-26,tour_finals,indoor,hard,...,1.16,5.00,1.220,5.010,1.25,5.25,1.19,4.57,2010_65_london_masters_cup,2010_65_london_masters_cup_2010-11-26_nadal_r_...
2675,tennis_data_uk,atp,2010,65,Masters Cup,London,2010-11-26,tour_finals,indoor,hard,...,1.38,3.00,1.480,2.920,1.48,3.15,1.40,2.88,2010_65_london_masters_cup,2010_65_london_masters_cup_2010-11-26_djokovic...
2676,tennis_data_uk,atp,2010,65,Masters Cup,London,2010-11-27,tour_finals,indoor,hard,...,1.50,2.62,1.530,2.740,1.65,2.75,1.52,2.50,2010_65_london_masters_cup,2010_65_london_masters_cup_2010-11-27_nadal_r_...
2677,tennis_data_uk,atp,2010,65,Masters Cup,London,2010-11-27,tour_finals,indoor,hard,...,1.38,3.00,1.420,3.180,1.50,3.24,1.40,2.92,2010_65_london_masters_cup,2010_65_london_masters_cup_2010-11-27_federer_...


In [314]:
csv_path = PROJECT_DIR / f"data/clean/tennis-data-uk/atp/uk_atp_singles_matches_{YEAR}.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_uk.to_csv(csv_path, index=False)
print(f"Written to {csv_path}")

Written to /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/clean/tennis-data-uk/atp/uk_atp_singles_matches_2010.csv
